# Using nfield as a tool in an agent

Agents call tools. A common one is "pull these fields out of this document." nfield fits that
slot: you wrap one extraction call in a function, describe it with a JSON Schema, and let the
model decide when to call it. This notebook builds a small, framework-free version of that loop
so the idea is clear without pulling in an agent library.

> This wraps nfield's existing extraction API as one tool an agent can call. It is a usage pattern, not a separate tool-calling feature: everything here is the public `NField` engine plus a plain function and a dispatcher.


## Setup

```bash
pip install "nfield[groq]"
export GROQ_API_KEY="gsk_..."
```

In [1]:
import json
import os

from nfield import NField

assert os.environ.get("GROQ_API_KEY"), "set GROQ_API_KEY first"

MODEL = "groq/llama-3.3-70b-versatile"

## The tool

The extraction schema is fixed for this tool: it pulls invoice fields. We build the engine once
and wrap it in a plain function. This is the function an agent would register as a tool.

In [2]:
INVOICE_SCHEMA = {
    "type": "object",
    "properties": {
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
        "due_date": {"type": "string"},
    },
    "required": ["vendor", "total"],
}

_engine = NField(MODEL, INVOICE_SCHEMA)


def extract_invoice(document: str) -> dict:
    """Pull structured invoice fields out of raw text."""
    return _engine.extract(document).data

## The tool description

An agent needs a machine-readable description of the tool: its name, what it does, and the shape
of its arguments. This is the same JSON Schema format tool-calling APIs expect.

In [3]:
TOOL_SPEC = {
    "name": "extract_invoice",
    "description": "Extract vendor, total, currency, and due date from raw invoice text.",
    "parameters": {
        "type": "object",
        "properties": {
            "document": {"type": "string", "description": "The raw invoice text."},
        },
        "required": ["document"],
    },
}

print(json.dumps(TOOL_SPEC, indent=2))

{
  "name": "extract_invoice",
  "description": "Extract vendor, total, currency, and due date from raw invoice text.",
  "parameters": {
    "type": "object",
    "properties": {
      "document": {
        "type": "string",
        "description": "The raw invoice text."
      }
    },
    "required": [
      "document"
    ]
  }
}


## The loop

A real agent hands `TOOL_SPEC` to a model, reads back a tool call, and dispatches it. Here we
stand in for that step: a small dispatcher maps the tool name to the function and runs it on the
document. Swap the dispatcher for your agent framework and the tool stays the same.

In [4]:
TOOLS = {"extract_invoice": extract_invoice}


def dispatch(tool_call: dict) -> dict:
    return TOOLS[tool_call["name"]](**tool_call["arguments"])


incoming_document = """
INVOICE #A-2231
Vendor: Globex Industrial
Amount Due: 4820.00 EUR
Payment due by: 2026-03-01
"""

# stands in for the tool call a model would emit
tool_call = {"name": "extract_invoice", "arguments": {"document": incoming_document}}

dispatch(tool_call)

{'vendor': 'Globex Industrial',
 'total': 4820.0,
 'currency': 'EUR',
 'due_date': '2026-03-01'}

## Why this is a good fit

The tool returns clean, typed JSON the agent can act on without parsing prose. Because nfield
validates each value against the document and retries the ones that fail, the tool is reliable
enough to sit inside a loop that makes decisions on its output. For wide documents with many
fields the same wrapper works unchanged; nfield handles the splitting.